# 第4章

In [ ]:
!pip install pgmpy==0.1.19
!pip install pandas==1.4.3

In [ ]:
from functools import partial
import numpy as np
import pandas as pd

data_url = "https://raw.githubusercontent.com/altdeep/causalAI/master/datasets/cigs_and_cancer.csv"
data = pd.read_csv(data_url)
cost_lower = np.quantile(data["C"], 1/3)
cost_upper = np.quantile(data["C"], 2/3)
def discretize_three(val, lower, upper):
    if val < lower:
        return "Low"
    if val < upper:
        return "Med"
    return "High"

data_disc = data.assign(
    C = lambda df: df['C'].map(
            partial(
                discretize_three,
                lower=cost_lower,
                upper=cost_upper
            )
        )
)
data_disc = data_disc.assign(
    L = lambda df: df['L'].map(str),
)
print(data_disc)

## リスト4.11

In [ ]:
from pgmpy.inference import VariableElimination
from pgmpy.models import NaiveBayes

model_L_given_CST = NaiveBayes()
model_L_given_CST.fit(data_disc, 'L')
infer_L_given_CST = VariableElimination(model_L_given_CST)

def p_L_given_CST(L_val, C_val, S_val, T_val):
    result_out = infer_L_given_CST.query(
        variables=["L"],
        evidence={'C': C_val, 'S': S_val, 'T': T_val},
        show_progress=False
    )
    var_outcomes = result_out.state_names["L"]
    var_values = result_out.values
    prob = dict(zip(var_outcomes, var_values))
    return prob[L_val]

## リスト4.12

In [ ]:
p_L_given_CST("True", "Low", "Low", "Low")

In [ ]:
model_S_given_C = NaiveBayes()
model_S_given_C.fit(data_disc, 'S')
infer_S_given_C = VariableElimination(model_S_given_C)
def p_S_given_C(S_val, C_val):
    result_out = infer_S_given_C.query(
        variables=['S'],
        evidence={'C': C_val},
        show_progress=False
    )
    var_names = result_out.state_names["S"]
    var_values = result_out.values
    prob = dict(zip(var_names, var_values))
    return prob[S_val]

## リスト4.13

In [ ]:
def h_function(L, C, T):
    summ = 0
    for s in ["Low", "Med", "High"]:
        summ += p_L_given_CST(L, C, s, T) * p_S_given_C(s, C)
    return summ

## リスト4.14

In [ ]:
ctl_outcomes = pd.DataFrame(
    [
        (C, T, L)
        for C in ["Low", "Med", "High"]
        for T in ["Low", "High"]
        for L in ["False", "True"]
    ],
    columns = ['C', 'T', 'L']
)

In [ ]:
print(ctl_outcomes)

## リスト4.15

In [ ]:
h_dist = ctl_outcomes.assign(
    h_func = ctl_outcomes.apply(
        lambda row: h_function(
            row['L'], row['C'], row['T']), axis = 1
    )
)
print(h_dist)

In [ ]:
df_mod = data_disc.merge(h_dist, on=['C', 'T', 'L'], how='left')
print(df_mod)

In [ ]:
df_mod.boxplot("h_func", "C")

In [ ]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

model = ols('h_func ~ C', data=df_mod).fit()
aov_table = sm.stats.anova_lm(model, typ=2)
aov_table["PR(>F)"]["C"]

In [ ]:
model = ols('h_func ~ T', data=df_mod).fit()
aov_table = sm.stats.anova_lm(model, typ=2)
aov_table["PR(>F)"]["T"]

In [ ]:
model = ols('h_func ~ L', data=df_mod).fit()
aov_table = sm.stats.anova_lm(model, typ=2)
aov_table["PR(>F)"]["L"]